# Klasifikasi gangguan jaringan Gedung Diskominfo Tangerang Selatan

# Import Library

In [ ]:
import os
import json
import warnings

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.preprocessing import MinMaxScaler

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
    learning_curve,
    cross_val_score
)

from sklearn.preprocessing import (
    StandardScaler,
    LabelEncoder
)

from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    auc,
    silhouette_score
)

import joblib

warnings.filterwarnings("ignore")

sns.set(style="darkgrid")

print("Library berhasil dimuat")

### SETUP FOLDER

In [ ]:
BASE_DIR = os.getcwd()

STATIC_DIR = os.path.join(BASE_DIR, "static")
MODELS_DIR = os.path.join(BASE_DIR, "models")

os.makedirs(STATIC_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

def save_plot(filename):
    plt.tight_layout()
    plt.savefig(
        os.path.join(STATIC_DIR, filename),
        dpi=150,
        bbox_inches="tight"
    )
    plt.close()

# Business Understanding

Dalam proses monitoring jaringan, data yang dihasilkan umumnya masih berbentuk log teknis yang sulit dipahami secara langsung oleh operator maupun pengambil keputusan. Data tersebut memuat berbagai informasi seperti throughput, delay komunikasi, jumlah paket, hingga durasi koneksi.

Permasalahan utama muncul ketika data yang sangat teknis tersebut harus diterjemahkan menjadi kondisi jaringan yang lebih mudah dipahami, misalnya apakah jaringan sedang berada dalam kondisi normal atau mengalami gangguan.

Melalui notebook ini, dilakukan pembangunan sistem klasifikasi kualitas jaringan menggunakan pendekatan machine learning dengan fokus pada empat parameter utama, yaitu bandwidth, latency, packet loss, dan uptime.

Model yang dibangun nantinya akan diintegrasikan ke dalam aplikasi berbasis Flask sehingga proses prediksi dapat dilakukan secara real-time melalui antarmuka web.

### Load Data

In [ ]:
data_path = os.path.join(BASE_DIR, "clear_data_kominfo.csv")

df_raw = pd.read_csv(data_path, sep=';')

print("Data berhasil dibaca")
print("Ukuran dataset:", df_raw.shape)

display(df_raw.head())

print("\nInfo dataset:")
df_raw.info()

### Data Understanding

Dataset yang digunakan merupakan data monitoring jaringan yang masih berada dalam kondisi mentah. Pada tahap awal ini dilakukan proses identifikasi struktur data untuk memahami karakteristik atribut yang tersedia sebelum masuk ke proses analisis lebih lanjut.

Tahap ini penting karena kualitas pemahaman terhadap data akan sangat mempengaruhi kualitas feature engineering maupun proses modeling yang dilakukan setelahnya.

In [ ]:
# Shape dataset
df_raw.shape

Jumlah observasi pada dataset menunjukkan bahwa data monitoring memiliki volume yang cukup untuk dilakukan proses pembelajaran model machine learning.

Banyaknya observasi akan membantu model dalam mempelajari pola performa jaringan secara lebih stabil.

In [ ]:
# Statistik deskriptif
df_raw.describe(include="all")

Statistik deskriptif memberikan gambaran awal mengenai pola distribusi data, sebaran nilai, serta potensi adanya nilai ekstrem pada beberapa atribut jaringan.

Tahap ini membantu dalam memahami perilaku dasar data sebelum dilakukan pembersihan dan transformasi lebih lanjut.

### DATA CLEANING

In [ ]:
def clean_number(x):
    try:
        return float(str(x).replace(".", "").replace(",", "."))
    except:
        return np.nan

In [ ]:
numeric_cols = [
    "bytes_per_second",
    "avg_inter_arrival_ms",
    "forward_packets",
    "backward_packets",
    "total_packets",
    "duration_seconds"
]

df = df_raw.copy()

for col in numeric_cols:
    df[col] = df[col].apply(clean_number)

Tahap pembersihan data dilakukan karena format numerik pada dataset masih belum konsisten. Beberapa nilai numerik masih tersimpan sebagai string sehingga tidak dapat langsung diproses oleh model machine learning.

Proses konversi dilakukan untuk memastikan seluruh atribut numerik dapat dianalisis secara matematis tanpa menyebabkan error pada tahap modeling.

In [ ]:
# Missing value
missing_values = df.isnull().sum()

missing_values

In [ ]:
plt.figure(figsize=(10,4))

sns.barplot(
    x=missing_values.index,
    y=missing_values.values
)

plt.xticks(rotation=45)
plt.title("Distribusi Missing Value")

plt.savefig("static\missing_values.png")
plt.show()

Nilai yang hilang ditangani menggunakan median karena metode ini lebih stabil terhadap outlier dibandingkan rata-rata.

Pendekatan ini dipilih untuk menjaga karakteristik distribusi data jaringan tetap mendekati kondisi aslinya.

# FEATURE ENGINEERING

In [ ]:
# Feature Engineering
df_fe = pd.DataFrame(index=df.index)


# BANDWIDTH

bw_scaler = MinMaxScaler(feature_range=(1, 1000))

df_fe["bandwidth"] = bw_scaler.fit_transform(
    df[["bytes_per_second"]]
).flatten()


# LATENCY

lat_scaler = MinMaxScaler(feature_range=(1, 300))

df_fe["latency"] = lat_scaler.fit_transform(
    df[["avg_inter_arrival_ms"]]
).flatten()


# PACKET LOSS

df_fe["packet_loss"] = (
    abs(
        df["forward_packets"]
        -
        df["backward_packets"]
    )
    /
    (df["total_packets"] + 1e-6)
) * 100


# UPTIME

up_scaler = MinMaxScaler(feature_range=(80, 100))

df_fe["uptime"] = up_scaler.fit_transform(
    df[["duration_seconds"]]
).flatten()

df_fe.head()

Keempat Variabel ini saya pilih sebagai paramter utama dalam perhitungan gangguan jaringan berdasarkan spesifikasi tolok ukur utama dalam pemilihan fitur yang menemukan label

In [ ]:
df_fe.info()

In [ ]:
print(df_fe.duplicated().sum())

In [ ]:
df_fe.drop_duplicates(inplace=True)

print(df_fe.shape)

In [ ]:
df_fe.isnull().sum()

### Histrogram

In [ ]:
features = [
    "bandwidth",
    "latency",
    "packet_loss",
    "uptime"
]

for col in features:

    plt.figure(figsize=(6,4))

    sns.histplot(
        df_fe[col],
        kde=True,
        bins=30
    )

    plt.title(f"Distribusi {col}")
    plt.show()
    save_plot(f"hist_{col}.png")

Histogram digunakan untuk memahami pola distribusi setiap fitur jaringan.

Melalui visualisasi ini dapat terlihat bahwa beberapa parameter memiliki distribusi yang tidak simetris. Dalam konteks jaringan, kondisi tersebut sering muncul akibat adanya lonjakan trafik atau ketidakstabilan komunikasi penggunaan jaringan pada waktu tertentu.

### Scaling Cluster

In [ ]:
scaler_cluster = StandardScaler()

X_cluster = scaler_cluster.fit_transform(
    df_fe[features]
)

### Clustering Menggunakan K-Means

In [ ]:
kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(X_cluster)

df_fe["cluster"] = clusters

df_fe.head()

Pendekatan clustering ini saya gunakan untuk membentuk pola alami pada data sebelum proses klasifikasi dilakukan.

Berbeda dengan pendekatan rule-based labeling langsung, clustering memungkinkan proses segmentasi data dilakukan berdasarkan kombinasi karakteristik jaringan secara keseluruhan. Pendekatan ini membantu mengurangi risiko data leakage dan overfitting pada model machine learning.

In [ ]:
sil_score = silhouette_score(
    X_cluster,
    clusters
)

print("Silhouette Score:", sil_score)

Silhouette Score digunakan untuk mengevaluasi kualitas pembentukan cluster. Nilai yang lebih tinggi menunjukkan bahwa masing-masing cluster memiliki pemisahan yang lebih baik dan tidak terlalu saling tumpang tindih.

Evaluasi clustering penting dilakukan untuk memastikan bahwa proses segmentasi data benar-benar membentuk pola yang bermakna dan mendapatkan insight.

In [ ]:
plt.figure(figsize=(7,5))

sns.scatterplot(
    data=df_fe,
    x="latency",
    y="packet_loss",
    hue="cluster",
    palette="Set1",
    alpha=0.7
)

plt.title("Hasil Clustering")
plt.show()
save_plot("cluster_scatter.png")

Visualisasi clustering memperlihatkan bahwa data jaringan mulai membentuk pola segmentasi berdasarkan karakteristik performa jaringan.

Pemisahan cluster ini menunjukkan bahwa data memiliki variasi perilaku yang cukup untuk digunakan dalam proses klasifikasi machine learning.

In [ ]:
cluster_summary = df_fe.groupby("cluster")[features].mean()

cluster_summary

In [ ]:
severity_df = df_fe.copy()

# Normalisasi 

severity_df["latency_score"] = (
    severity_df["latency"]
    /
    severity_df["latency"].max()
) * 100

severity_df["packet_loss_score"] = (
    severity_df["packet_loss"]
    /
    severity_df["packet_loss"].max()
) * 100

severity_df["uptime_score"] = (
    100 -
    (
        severity_df["uptime"]
        /
        severity_df["uptime"].max()
    ) * 100
)

severity_df["bandwidth_score"] = (
    100 -
    (
        severity_df["bandwidth"]
        /
        severity_df["bandwidth"].max()
    ) * 100
)

### Final Serevity Score

In [ ]:
severity_df["severity_score"] = (

    (severity_df["latency_score"] * 0.40)

    +

    (severity_df["packet_loss_score"] * 0.40)

    +

    (severity_df["uptime_score"] * 0.15)

    +

    (severity_df["bandwidth_score"] * 0.05)

)

severity_df[[
    "bandwidth",
    "latency",
    "packet_loss",
    "uptime",
    "severity_score"
]].head()

Severity score dibangun menggunakan kombinasi beberapa parameter performa jaringan dengan bobot yang berbeda sesuai tingkat pengaruhnya terhadap kualitas jaringan.

Latency dan packet loss diberikan bobot lebih besar karena kedua parameter tersebut memiliki pengaruh paling signifikan terhadap stabilitas komunikasi jaringan.

Sementara itu bandwidth dan uptime tetap digunakan sebagai parameter pendukung untuk membantu model memahami kondisi jaringan secara lebih menyeluruh.

In [ ]:
plt.figure(figsize=(10,5))

sns.histplot(
    severity_df["severity_score"],
    bins=30,
    kde=True
)

plt.title("Distribusi Severity Score")
plt.xlabel("Severity Score")
plt.ylabel("Frekuensi")
plt.show()

Grafik menunjukkan persebaran nilai severity score yang digunakan sebagai dasar pembentukan label gangguan jaringan. Distribusi yang baik menunjukkan variasi kondisi jaringan dari normal hingga gangguan.

In [ ]:
severity_df.groupby("cluster")[[
    "bandwidth",
    "latency",
    "packet_loss",
    "uptime",
    "severity_score"
]].mean()

### Labeling Menggunakan Quantile dan Berdasarkan Severity Score

Dari Hasil Clustering

In [ ]:
np.random.seed(42)

# Tambahkan jitter proporsional ke severity_score
# untuk menyebarkan mass point di nilai 60.0
_noise_scale = severity_df["severity_score"].std() * 0.35
_severity_jittered = (
    severity_df["severity_score"]
    + np.random.normal(0, _noise_scale, len(severity_df))
)

# Threshold berdasarkan distribusi yang sudah di-jitter
q_low  = _severity_jittered.quantile(0.35)
q_high = _severity_jittered.quantile(0.65)

labels = []

for score in _severity_jittered:

    # Zona Bawah → Normal
    if score <= q_low:
        labels.append("normal")

    # Zona Atas → Gangguan
    elif score >= q_high:
        labels.append("gangguan")

    # Zona Transisi — ambiguous boundary
    else:
        midpoint = (q_low + q_high) / 2
        if score >= midpoint:
            labels.append("gangguan")
        else:
            labels.append("normal")

# Finalisasi Label
severity_df["label"] = labels

# Distribusi Label
print(severity_df["label"].value_counts())
print(severity_df["label"].value_counts(normalize=True).round(3))

In [ ]:
severity_df.head()

In [ ]:
severity_df[[
    "latency",
    "packet_loss",
    "uptime",
    "bandwidth",
    "severity_score"
]].corr()

In [ ]:
severity_df[[
    "bandwidth",
    "latency",
    "packet_loss",
    "uptime"
]].describe().T

In [ ]:
severity_df[[
    "bandwidth",
    "latency",
    "packet_loss",
    "uptime"
]].head()

In [ ]:
labeled_df = severity_df[
    [
        "bandwidth",
        "latency",
        "packet_loss",
        "uptime",
        "label"
    ]
].copy()

labeled_df.head()

In [ ]:
labeled_df = labeled_df.copy()

# uptime 0-1 menjadi %
if labeled_df["uptime"].max() <= 1:
    labeled_df["uptime"] = labeled_df["uptime"] * 100

# packet loss 0-1 menjadi %
if labeled_df["packet_loss"].max() <= 1:
    labeled_df["packet_loss"] = labeled_df["packet_loss"] * 100

labeled_df["bandwidth"] = labeled_df["bandwidth"].round(2)
labeled_df["latency"] = labeled_df["latency"].round(2)
labeled_df["packet_loss"] = labeled_df["packet_loss"].round(2)
labeled_df["uptime"] = labeled_df["uptime"].round(2)

In [ ]:
comparison_df = pd.DataFrame({
    "Feature": ["bandwidth", "latency", "packet_loss", "uptime"],
    "Min": [
        labeled_df["bandwidth"].min(),
        labeled_df["latency"].min(),
        labeled_df["packet_loss"].min(),
        labeled_df["uptime"].min()
    ],
    "Max": [
        labeled_df["bandwidth"].max(),
        labeled_df["latency"].max(),
        labeled_df["packet_loss"].max(),
        labeled_df["uptime"].max()
    ]
})

comparison_df

In [ ]:
labeled_df.head()

In [ ]:
labeled_df.to_csv('data\labeled_data.csv', index=False)
print("Dataset dengan label berhasil disimpan ke 'labeled_data.csv'")

Tahap interpretasi cluster dilakukan untuk memahami karakteristik masing-masing kelompok data jaringan.

Cluster dengan latency lebih tinggi, packet loss lebih besar, dan bandwidth lebih rendah akan diinterpretasikan sebagai kondisi gangguan. Sementara cluster dengan performa jaringan lebih stabil akan dikategorikan sebagai kondisi normal.

# Data Preparation 

### Exploratory Analysis Data Distribusi Label

In [ ]:
plt.figure(figsize=(6,5))

ax = sns.countplot(
    data=labeled_df,
    x="label"
)

for p in ax.patches:
    ax.annotate(
        f"{int(p.get_height())}",
        (p.get_x()+0.3, p.get_height()+10)
    )

plt.title("Distribusi Label Gangguan Jaringan")
plt.xlabel("Label")
plt.ylabel("Jumlah Data")
plt.show()

### Boxplot


In [ ]:
for col in features:

    plt.figure(figsize=(6,5))

    sns.boxplot(x=df_fe[col])

    plt.title(f"Boxplot {col}")
    plt.show()

    save_plot(f"boxplot_{col}.png")

Visualisasi boxplot memperlihatkan adanya beberapa nilai ekstrem pada sejumlah parameter jaringan.

Nilai ekstrem pada data capturing tidak selalu dianggap sebagai noise. Dalam beberapa kasus, kondisi tersebut justru dapat merepresentasikan gangguan jaringan yang benar-benar terjadi pada lingkungan gedung kantor operasional.

### Heatmap korelasi

In [ ]:
plt.figure(figsize=(6,5))

corr = labeled_df[features].corr()

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Korelasi Antar Fitur")
plt.show()
save_plot("correlation_heatmap.png")

Heatmap korelasi digunakan untuk memahami hubungan antar parameter jaringan.

Hubungan antar fitur penting untuk dianalisis karena beberapa algoritma machine learning memiliki sensitivitas terhadap fitur yang saling berkorelasi tinggi. Selain itu, korelasi juga membantu memahami perilaku performa jaringan secara lebih menyeluruh.

### KDE berdasarkan label

In [ ]:
for col in features:

    plt.figure(figsize=(6,4))

    sns.kdeplot(
        data=labeled_df,
        x=col,
        hue="label",
        common_norm=False
    )

    plt.title(f"{col} berdasarkan Label")
    plt.show()

    save_plot(f"kde_{col}.png")

Distribusi fitur berdasarkan label memperlihatkan bahwa kondisi normal dan gangguan memiliki pola sebaran yang berbeda pada beberapa parameter jaringan.

Perbedaan pola tersebut menjadi indikasi bahwa fitur yang dibentuk memiliki kemampuan untuk membantu model dalam membedakan kondisi jaringan secara lebih efektif.

### Scatter plot 1

In [ ]:
plt.figure(figsize=(6,5))

sns.scatterplot(
    data=labeled_df,
    x="latency",
    y="packet_loss",
    hue="label",
    alpha=0.7
)

plt.title("Latency vs Packet Loss")
plt.show()
save_plot("scatter_latency_packetloss.png")

Scatter plot memperlihatkan bahwa peningkatan latency sering kali diikuti dengan kenaikan packet loss pada sejumlah observasi jaringan.

Pola ini menunjukkan bahwa keterlambatan komunikasi dan ketidakstabilan transmisi data memiliki hubungan yang cukup erat dalam merepresentasikan kondisi gangguan jaringan.

### Scatter plot 2

In [ ]:
plt.figure(figsize=(6,5))

sns.scatterplot(
    data=labeled_df,
    x="bandwidth",
    y="latency",
    hue="label",
    alpha=0.7
)

plt.title("Bandwidth vs Latency")
plt.show()
save_plot("scatter_bandwidth_latency.png")

Visualisasi bandwidth dan latency menunjukkan bahwa koneksi dengan bandwidth rendah cenderung memiliki latency yang lebih tinggi dibandingkan koneksi yang stabil.

Fenomena ini menggambarkan bagaimana keterbatasan throughput dapat mempengaruhi kualitas komunikasi jaringan secara keseluruhan.

### Scatter plot 3


In [ ]:
plt.figure(figsize=(6,5))

sns.scatterplot(
    data=labeled_df,
    x="uptime",
    y="packet_loss",
    hue="label",
    alpha=0.7
)

plt.title("Uptime vs Packet Loss")
plt.show()
save_plot("scatter_uptime_packetloss.png")

# DATA PREPARATION UNTUK MODELING

### Label Encoder

In [ ]:
features = [
    "bandwidth",
    "latency",
    "packet_loss",
    "uptime"
]
x = labeled_df[features]

y = labeled_df["label"]

le = LabelEncoder()

y_enc = le.fit_transform(y)


print(y.value_counts())

### Split training & test

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y_enc,
    test_size=0.2,
    stratify=y_enc,
    random_state=42
)

### Scaler

In [ ]:
scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

# Hyperparameter Tuning 

### Tuning Random Forest

In [ ]:
param_grid = {

    "n_estimators":[
        100,
        200,
        300
    ],

    "max_depth":[
        5,
        10,
        15
    ],

    "min_samples_split":[
        5,
        10,
        20
    ],

    "min_samples_leaf":[
        2,
        5,
        10
    ]
}

grid_rf = GridSearchCV(

    RandomForestClassifier(
        random_state=42
    ),

    param_grid=param_grid,

    cv=5,

    scoring="f1_macro",

    n_jobs=-1

)

grid_rf.fit(
    x_train,
    y_train
)

rf = grid_rf.best_estimator_

print(
    grid_rf.best_params_
)

Grid Search digunakan untuk mencari kombinasi parameter terbaik pada algoritma Random Forest sehingga mampu menghasilkan performa yang optimal pada data pelatihan.

### Hasil Tabel Kombinasi Parameter

In [ ]:
rf_tuning_results = pd.DataFrame(
    grid_rf.cv_results_
)

rf_tuning_results[
    [
        "mean_test_score",
        "std_test_score",
        "param_n_estimators",
        "param_max_depth",
        "param_min_samples_split",
        "param_min_samples_leaf"
    ]
].sort_values(
    by="mean_test_score",
    ascending=False
).head(10)

Tabel berikut menampilkan kombinasi parameter terbaik berdasarkan nilai rata-rata F1-Score hasil Cross Validation. Kombinasi parameter dengan nilai F1-Score tertinggi dipilih sebagai model Random Forest terbaik yang akan digunakan pada tahap evaluasi.

### Visualisasi Hasil Tuning Random Forest

In [ ]:
top10_rf = rf_tuning_results.sort_values(
    by="mean_test_score",
    ascending=False
).head(10)

plt.figure(figsize=(12,6))

sns.barplot(
    data=top10_rf,
    x="mean_test_score",
    y=top10_rf.index,
)

plt.xlabel("Mean F1 Score")
plt.ylabel("Kombinasi Parameter")
plt.title("Top 10 Hyperparameter Random Forest")

plt.show()

### Tuning Naive Bayes

In [ ]:
param_grid_nb = {

    "var_smoothing": [
        1e-12,
        1e-11,
        1e-10,
        1e-9,
        1e-8,
        1e-7,
        1e-6,
        1e-5,
        1e-4,
        1e-3
    ]

}

grid_nb = GridSearchCV(

    estimator=GaussianNB(),

    param_grid=param_grid_nb,

    cv=5,

    scoring="f1_macro",

    n_jobs=-1

)

grid_nb.fit(
    x_train,
    y_train
)

nb = grid_nb.best_estimator_

print("Best Parameter:")
print(grid_nb.best_params_)

print("Best Score:")
print(grid_nb.best_score_)

Proses hyperparameter tuning dilakukan menggunakan Grid Search Cross Validation untuk mencari nilai parameter `var_smoothing` terbaik pada algoritma Gaussian Naive Bayes.

Parameter `var_smoothing` digunakan untuk menambahkan nilai stabilisasi pada varians setiap fitur sehingga dapat mengurangi risiko overfitting dan meningkatkan kemampuan generalisasi model terhadap data yang belum pernah dilihat sebelumnya.

Nilai parameter terbaik dipilih berdasarkan skor F1-Macro tertinggi pada proses validasi silang (Cross Validation).

### Visualisasi Hasil Tuning Naive Bayes

In [ ]:
cv_nb = pd.DataFrame(
    grid_nb.cv_results_
)

plt.figure(figsize=(10,5))

plt.semilogx(
    cv_nb["param_var_smoothing"].astype(float),
    cv_nb["mean_test_score"],
    marker="o"
)

plt.xlabel("var_smoothing")

plt.ylabel("Mean F1 Macro")

plt.title(
    "Hyperparameter Tuning Gaussian Naive Bayes"
)

plt.grid(True)

plt.show()

Data Preparation

Dataset ini dipisahkan menjadi data training dan testing agar proses evaluasi dapat dilakukan secara objektif.

Selain itu, dilakukan proses scaling untuk menyeimbangkan rentang nilai antar fitur sehingga model dapat mempelajari pola data secara lebih optimal.

# Modeling 


### Implementasi Algoritma Random Forest

In [ ]:
param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [3, 5, 7],
    "min_samples_split": [10, 20, 30],
    "min_samples_leaf": [5, 10, 15],
    "max_features": ["sqrt"]
}

grid_rf = GridSearchCV(
    estimator=RandomForestClassifier(
        random_state=42
    ),

    param_grid=param_grid,

    cv=5,

    scoring="f1",

    n_jobs=-1
)

grid_rf.fit(x_train, y_train)

rf = grid_rf.best_estimator_

print("Random Forest:")
print(grid_rf.best_params_)

### Implementasi Algoritma Naive Bayes

In [ ]:
nb = grid_nb.best_estimator_

nb.fit(x_train, y_train)

Model Random Forest dioptimasi menggunakan GridSearchCV untuk mencari kombinasi parameter terbaik berdasarkan nilai F1-score.

Sementara itu, Naive Bayes digunakan sebagai pembanding karena memiliki karakteristik model probabilistik yang ringan dan cepat dalam proses komputasi.

# Evaluasi Model

In [ ]:
y_pred_rf = rf.predict(x_test)
y_pred_nb = nb.predict(x_test)

### Cross Validasi

In [ ]:
rf_cv = cross_val_score(
    rf,
    x_train,
    y_train,
    cv=5,
    scoring="f1"
)

nb_cv = cross_val_score(
    nb,
    x_train,
    y_train,
    cv=5,
    scoring="f1"
)

print("RF CV Mean:", rf_cv.mean())
print("NB CV Mean:", nb_cv.mean())

### Hasil Nilai Klasifikasi

In [ ]:
print("=== RANDOM FOREST ===")
print(
    classification_report(
        y_test,
        y_pred_rf,
        target_names=le.classes_
    )
)

print("=== NAIVE BAYES ===")
print(
    classification_report(
        y_test,
        y_pred_nb,
        target_names=le.classes_
    )
)

Evaluasi dilakukan menggunakan beberapa metrik seperti accuracy, precision, recall, dan F1-score. sebagai model klasifikasi

Dalam konteks monitoring jaringan, recall menjadi salah satu metrik yang penting karena model diharapkan mampu mendeteksi kondisi gangguan sebanyak mungkin untuk meminimalkan risiko gangguan yang tidak terdeteksi.

### Confusion Matrix 


In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred_rf
)

plt.figure(figsize=(7,6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    linewidths=1,
    square=True
)

plt.title(
    "Confusion Matrix Random Forest",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel(
    "Predicted Label"
)

plt.ylabel(
    "Actual Label"
)

plt.show()

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred_nb
)

plt.figure(figsize=(7,6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Reds",
    cbar=False,
    linewidths=1,
    square=True
)

plt.title(
    "Confusion Matrix Naive Bayes",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel(
    "Predicted Label"
)

plt.ylabel(
    "Actual Label"
)

plt.show()

Confusion matrix membantu memperlihatkan jenis kesalahan prediksi yang dilakukan model.

Kesalahan yang paling perlu diperhatikan adalah ketika kondisi gangguan diprediksi sebagai normal karena kondisi tersebut dapat menyebabkan gangguan jaringan tidak terdeteksi lebih awal.

### ROC Curve

In [ ]:
# ROC Curve
gangguan_index = int(le.transform(["gangguan"])[0])

rf_proba = rf.predict_proba(x_test)[:, gangguan_index]
nb_proba = nb.predict_proba(x_test)[:, gangguan_index]

y_test_bin = (
    y_test == gangguan_index
).astype(int)

fpr_rf, tpr_rf, _ = roc_curve(
    y_test_bin,
    rf_proba
)

fpr_nb, tpr_nb, _ = roc_curve(
    y_test_bin,
    nb_proba
)

roc_auc_rf = auc(fpr_rf, tpr_rf)
roc_auc_nb = auc(fpr_nb, tpr_nb)

plt.figure(figsize=(6,5))
plt.show()
plt.plot(
    fpr_rf,
    tpr_rf,
    label=f"Random Forest AUC={roc_auc_rf:.3f}"
)

plt.plot(
    fpr_nb,
    tpr_nb,
    label=f"Naive Bayes AUC={roc_auc_nb:.3f}"
)

plt.plot([0,1],[0,1],"k--")

plt.legend()

plt.title("ROC Curve")
plt.show()
save_plot("roc_curve.png")

ROC Curve memperlihatkan kemampuan model dalam membedakan antara kondisi normal dan gangguan pada berbagai threshold probabilitas.

Semakin mendekati sudut kiri atas, semakin baik kemampuan model dalam melakukan klasifikasi.

### Precision Recall Curve

In [ ]:
# Precision Recall Curve
prec_rf, rec_rf, _ = precision_recall_curve(
    y_test_bin,
    rf_proba
)

prec_nb, rec_nb, _ = precision_recall_curve(
    y_test_bin,
    nb_proba
)

plt.figure(figsize=(6,4))

plt.plot(
    rec_rf,
    prec_rf,
    label="Random Forest"
)

plt.plot(
    rec_nb,
    prec_nb,
    label="Naive Bayes"
)

plt.xlabel("Recall")
plt.ylabel("Precision")

plt.legend()

plt.title("Precision Recall Curve")
plt.show()
save_plot("pr_curve.png")

Precision Recall Curve digunakan untuk melihat keseimbangan antara kemampuan model dalam mendeteksi gangguan dan ketepatan prediksi yang dihasilkan.

Visualisasi ini menjadi penting terutama ketika distribusi kelas tidak sepenuhnya seimbang.

### Tabel Perbandingan Model

In [ ]:
metrics_df = pd.DataFrame({
    "Model": ["Random Forest", "Naive Bayes"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_nb)
    ],
    "Precision": [
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_nb)
    ],
    "Recall": [
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_nb)
    ],
    "F1-Score": [
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_nb)
    ]
})

metrics_df

### Menyimpan Hasil Klasifikasi Ke Dalam JSON

In [ ]:
# Export Metrics Summary Ke JSON

rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)

nb_accuracy = accuracy_score(y_test, y_pred_nb)
nb_precision = precision_score(y_test, y_pred_nb)
nb_recall = recall_score(y_test, y_pred_nb)
nb_f1 = f1_score(y_test, y_pred_nb)

metrics_summary = {
    "rf": {
        "accuracy": float(rf_accuracy),
        "precision": float(rf_precision),
        "recall": float(rf_recall),
        "f1": float(rf_f1),
        "roc_auc": float(roc_auc_rf),
        "pr_auc": float(
            auc(rec_rf, prec_rf)
        )
    },

    "nb": {
        "accuracy": float(nb_accuracy),
        "precision": float(nb_precision),
        "recall": float(nb_recall),
        "f1": float(nb_f1),
        "roc_auc": float(roc_auc_nb),
        "pr_auc": float(
            auc(rec_nb, prec_nb)
        )
    },

    "label_classes": list(le.classes_),

    "feature_names": [
        "bandwidth",
        "latency",
        "packet_loss",
        "uptime"
    ]
}

with open(
    os.path.join(BASE_DIR, "metrics_summary.json"),
    "w"
) as f:

    json.dump(
        metrics_summary,
        f,
        indent=4
    )

print("metrics_summary.json berhasil diperbarui")

### Grafik Bar Perbandingan Kedua Algoritma

In [ ]:
metrics_df.set_index("Model").plot(
    kind="bar",
    figsize=(8,5)
)

plt.title("Perbandingan Performa Model")
plt.show()
save_plot("model_comparison.png")

Perbandingan performa model menunjukkan bahwa masing-masing algoritma memiliki karakteristik yang berbeda dalam memahami pola data jaringan.

Random Forest cenderung menghasilkan performa yang lebih stabil pada berbagai metrik evaluasi, sedangkan Naive Bayes memiliki kecenderungan menghasilkan recall yang tinggi namun lebih sensitif terhadap distribusi data.

### Learning Curve

In [ ]:
from pathlib import Path

RANDOM_STATE = 42

def generate_learning_curve(estimator, X_vec, y, cv=5, train_sizes=np.linspace(0.1, 1.0, 10)):
    """
    Menghasilkan data learning curve.
    Return:
        train_sizes_abs, train_mean, train_std, val_mean, val_std, gap, val_final
    """
    train_sizes_abs, train_scores, val_scores = learning_curve(
        estimator,
        X_vec,
        y,
        train_sizes=train_sizes,
        cv=cv,
        scoring="f1_macro",
        n_jobs=-1,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    train_mean = train_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    val_mean   = val_scores.mean(axis=1)
    val_std    = val_scores.std(axis=1)

    gap = train_mean[-1] - val_mean[-1]
    val_final = val_mean[-1]

    return train_sizes_abs, train_mean, train_std, val_mean, val_std, gap, val_final


def diagnosa_fit(gap, val_final):
    """
    Diagnosis sederhana berdasarkan gap dan validation score.
    """
    if gap > 0.20:
        return "OVERFITTING", "#e74c3c"
    elif val_final < 0.45 and gap < 0.05:
        return "UNDERFITTING", "#e67e22"
    elif gap <= 0.10 and val_final >= 0.60:
        return "GOOD FIT", "#27ae60"
    else:
        return "SLIGHT OVERFIT", "#f39c12"


def plot_single_learning_curve(estimator, X_vec, y, nama_model, filename):
    """
    Plot learning curve satu model dalam satu figure tersendiri.
    Output PNG disimpan ke folder img/.
    """
    print(f"Menghitung learning curve: {nama_model}...")

    ts, tm, ts_std, vm, vs_std, gap, val_f = generate_learning_curve(
        estimator, X_vec, y
    )
    diagnosis, diag_color = diagnosa_fit(gap, val_f)

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.plot(
        ts, tm,
        color="steelblue",
        marker="o",
        markersize=5,
        linewidth=2,
        label="Training Score"
    )
    ax.plot(
        ts, vm,
        color="tomato",
        marker="s",
        markersize=5,
        linewidth=2,
        label="Validation Score (CV-5)"
    )

    ax.fill_between(ts, tm - ts_std, tm + ts_std, alpha=0.12, color="steelblue")
    ax.fill_between(ts, vm - vs_std, vm + vs_std, alpha=0.12, color="tomato")

    ax.axhline(
        y=0.80,
        color="#27ae60",
        linestyle="--",
        linewidth=1.5,
        alpha=0.7,
        label="Target 80%"
    )

    ax.annotate(
        f"Gap = {gap:.3f}",
        xy=(ts[-1], (tm[-1] + vm[-1]) / 2),
        xytext=(ts[-3], (tm[-1] + vm[-1]) / 2 + 0.05),
        fontsize=9,
        color="gray",
        arrowprops=dict(arrowstyle="->", color="gray", lw=1)
    )

    ax.set_title(
        f"{nama_model}\nGap={gap:.3f} | Val F1-Mac={val_f:.3f} | [{diagnosis}]",
        fontsize=12,
        fontweight="bold",
        color=diag_color,
        pad=12
    )
    ax.set_xlabel("Jumlah Sampel Training", fontsize=11)
    ax.set_ylabel("F1-Score Macro", fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.legend(loc="lower right", fontsize=9)
    ax.grid(alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    Path("static").mkdir(exist_ok=True)
    save_path = Path("static") / filename

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Disimpan: {filename}")
    print(f"Diagnosis: {diagnosis} | Val={val_f:.4f} | Gap={gap:.4f}\n")

    return gap, val_f, diagnosis

Learning curve digunakan untuk memahami bagaimana model belajar terhadap data training dan bagaimana kemampuan generalisasinya terhadap data validasi.

Apabila jarak antara kurva training dan validation terlalu jauh, maka model berpotensi mengalami overfitting. Sebaliknya, apabila kedua kurva sama-sama rendah, maka model dapat mengalami underfitting.

Visualisasi ini membantu memastikan bahwa model tidak hanya menghafal data training, tetapi juga mampu memahami pola yang lebih umum pada data jaringan.

In [ ]:
results = {}

results["Random Forest"] = plot_single_learning_curve(
    rf,
    x_train,
    y_train,
    "Random Forest",
    "lc_random_forest.png"
)

results

In [ ]:
results["Naive Bayes"] = plot_single_learning_curve(
    nb,
    x_train,
    y_train,
    "Naive Bayes",
    "lc_naive_bayes.png"
)

results

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    rf,
    x_train,
    y_train,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
val_mean = val_scores.mean(axis=1)

plt.figure(figsize=(7,5))

plt.plot(
    train_sizes,
    train_mean,
    label="Training Score"
)

plt.plot(
    train_sizes,
    val_mean,
    label="Validation Score"
)

plt.xlabel("Jumlah Data Training")
plt.ylabel("Accuracy")

plt.title("Learning Curve")
plt.show()
plt.legend()

save_plot("learning_curve.png")

Berdasarkan grafik learning curve, model Random Forest menunjukkan performa yang stabil dengan nilai training score dan validation score yang relatif berdekatan. Pada jumlah data pelatihan terbesar, model memperoleh training score sebesar 81,6% dan validation score sebesar 81,0% dengan selisih (gap) sekitar 0,6%.

Perbedaan yang kecil antara kedua kurva menunjukkan bahwa model tidak mengalami overfitting maupun underfitting yang signifikan. Selain itu, kurva validasi cenderung stabil ketika jumlah data meningkat, yang mengindikasikan bahwa model memiliki kemampuan generalisasi yang baik terhadap data yang belum pernah dilihat sebelumnya.

Dengan demikian, model Random Forest dapat dikategorikan sebagai model yang memiliki karakteristik good fit dan cukup representatif untuk digunakan dalam proses klasifikasi gangguan jaringan.

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    nb,
    x_train,
    y_train,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
val_mean = val_scores.mean(axis=1)

plt.figure(figsize=(7,5))

plt.plot(
    train_sizes,
    train_mean,
    label="Training Score"
)

plt.plot(
    train_sizes,
    val_mean,
    label="Validation Score"
)

plt.xlabel("Jumlah Data Training")
plt.ylabel("Accuracy")

plt.title("Learning Curve")
plt.show()
plt.legend()

save_plot("learning_curve.png")

Berdasarkan grafik learning curve, model Naive Bayes menunjukkan performa yang stabil seiring bertambahnya jumlah data pelatihan. Nilai training score dan validation score cenderung konvergen pada rentang 0,79–0,80 dengan gap yang sangat kecil.

Kondisi tersebut menunjukkan bahwa model tidak mengalami overfitting maupun underfitting yang signifikan. Meskipun terjadi sedikit penurunan pada validation score ketika jumlah data bertambah, perbedaannya relatif kecil dan mengindikasikan bahwa model mampu melakukan generalisasi dengan baik terhadap data yang belum pernah dilihat sebelumnya.

Karakteristik ini sesuai dengan sifat algoritma Naive Bayes yang cenderung menghasilkan model sederhana, stabil, dan memiliki risiko overfitting yang rendah.


# Feature Importances


In [ ]:
importance = rf.feature_importances_

fi_df = pd.DataFrame({
    "Feature": features,
    "Importance": importance
}).sort_values(
    by="Importance",
    ascending=False
)

fi_df

In [ ]:
plt.figure(figsize=(7,4))

sns.barplot(
    data=fi_df,
    x="Importance",
    y="Feature"
)

plt.title("Feature Importance")
plt.show()
save_plot("feature_importance.png")

Feature importance memperlihatkan kontribusi masing-masing parameter jaringan terhadap proses klasifikasi.

Melalui analisis ini dapat diketahui parameter mana yang paling berpengaruh dalam membedakan kondisi normal dan gangguan pada jaringan.

# Menyimpan Model Hasil Training (Deployment)

In [ ]:
# Save Model  Hasil Training Menggunakan Joblib 

joblib.dump(
    rf,
    os.path.join(MODELS_DIR, "rf_model.pkl")
)

joblib.dump(
    nb,
    os.path.join(MODELS_DIR, "nb_model.pkl")
)

joblib.dump(
    scaler,
    os.path.join(MODELS_DIR, "scaler.pkl")
)

joblib.dump(
    le,
    os.path.join(MODELS_DIR, "label_encoder.pkl")
)